# 第 19 章 · 仿真与部署

本章回答：同一套导航 / 机械臂代码，怎么先在仿真里跑通、再一键切到真机？Gazebo、MuJoCo、Isaac Sim 各接在 ROS 2 的哪一层？写好的系统怎么打包、装到机器人上、开机自启？Wi-Fi 下传大消息为什么会"卡死 30 秒"？

配套讲义：仓库里 `ros2/notes/lessons/19-sim-deploy.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 ROS 2。

**为什么是模拟**：这里不需要 ROS，只是算术。30 秒、256 KB 两个数来自 DDS-tuning.rst L47–L48；64 KB 一个 UDP 包、1500 字节的 MTU（每片约 1480 字节数据）是常见网络默认值，不是 ROS 文档里的数；丢包率是编的。

In [ ]:
import math

print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 规则 / 数字 | 写法 | 跑出来 |
|---|---|---|---|
| 第 3 步 分片 | 大消息拆成很多 IP 分片 | `fragments_per_datagram()` | 每个 64 KB 的 UDP 包 45 片 |
| 同上 | 一张 1080p 原始图 | `ceil(6,220,800 / 65,507)` | 95 个 UDP 包，约 4275 片 |
| 第 3 步 丢一片全包作废 | $1-(1-p)^n$ | `datagram_loss(0.01, 45)` | 约 36% |
| 第 3 步 缓冲区占满 | 256 KB、等 30 秒 | `incomplete_datagrams_to_fill_buffer()` | 约 4 个残缺包就占满 |

## 1. 先把要用的东西定义好

这几格只是定义（常量、函数、类），运行后没有输出也正常。用到的前面章节的函数也一起抄进来了，不需要本地文件。

**常量 `IPFRAG_TIME_S`**

In [ ]:
IPFRAG_TIME_S = 30  # Linux 默认等分片重组的时间（DDS-tuning.rst L47）

**常量 `UDP_MAX`**

In [ ]:
UDP_MAX = 65_507  # 一个 UDP 包最多带的数据（字节）

**函数 `datagram_loss`**：任何一片丢了，整个 UDP 包就拼不回来：1 − (1 − p)^n。

In [ ]:
def datagram_loss(p_fragment: float, n_fragments: int) -> float:
    """任何一片丢了，整个 UDP 包就拼不回来：1 − (1 − p)^n。"""
    return 1 - (1 - p_fragment) ** n_fragments

**常量 `IP_PAYLOAD`**

In [ ]:
IP_PAYLOAD = 1_480  # MTU 1500 减去 20 字节 IP 头

**函数 `fragments_per_datagram`**：一个 UDP 包被拆成几片 IP 分片。

In [ ]:
def fragments_per_datagram(size: int = UDP_MAX) -> int:
    """一个 UDP 包被拆成几片 IP 分片。"""
    return math.ceil(size / IP_PAYLOAD)

**常量 `IPFRAG_BUFFER`**

In [ ]:
IPFRAG_BUFFER = 262_144  # 分片重组缓冲区默认 256 KB（DDS-tuning.rst L48、L75）

**函数 `incomplete_datagrams_to_fill_buffer`**：每个残缺的 UDP 包最多占着接近 size 字节，几个就能把 256 KB 缓冲区占满。

In [ ]:
def incomplete_datagrams_to_fill_buffer(size: int = UDP_MAX) -> float:
    """每个残缺的 UDP 包最多占着接近 size 字节，几个就能把 256 KB 缓冲区占满。"""
    return IPFRAG_BUFFER / size

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第三步 · 一张 1080p 原始图像（第 10 章：6,220,800 字节）要拆成多少片")
image = 1920 * 1080 * 3
n_udp = math.ceil(image / UDP_MAX)
n_frag = fragments_per_datagram()
print(f"  {n_udp} 个 UDP 包，每个 {n_frag} 片 IP 分片，一共约 {n_udp * n_frag} 片")

In [ ]:
print("\n第三步（续）· 每片只丢 1%（编的），一个 64 KB 的 UDP 包有多大概率拼不回来")
print(f"  {datagram_loss(0.01, n_frag):.0%}")

In [ ]:
print("\n第三步（再续）· 拼不回来的残片占着缓冲区：多少个就占满 256 KB，占满后要等多久")
print(f"  约 {incomplete_datagrams_to_fill_buffer():.0f} 个就满；默认要等 {IPFRAG_TIME_S} 秒才清掉")

## 3. 代码里才看得到的

1. **30 秒和 256 KB 来自 DDS-tuning.rst L47–L48**；64 KB 的 UDP 上限、1500 字节 MTU 是常见网络默认值，不是 ROS 文档里的数；1% 的丢包率是编的。
2. **$(1-p)^n$ 是"n 片都到"的概率**：每片独立地以 1−p 的概率到达，n 片都到才算这个包到了。1% 看起来很小，乘 45 次就剩 64%。
3. **DDS 自己也会切片**：很多 DDS 实现会把大消息先切成小于 64 KB 的块再交给 UDP，具体大小看中间件配置。

## 4. 自己改一改

- 把丢包率改成 0.1%，看整包丢失的概率；
- 把消息换成压缩后的 200 KB 图像，算要几个 UDP 包、几片。

在下面的空格子里写，或者直接改上面的格子。讲义末尾的练习先在纸上做，不在这里给答案。

In [ ]:
# 在这里写你的代码

## 口诀

- **仿真到真机**：上层不动，换 `<plugin>`、关 `use_sim_time`、换传感器来源。
- **仿真器**：Gazebo 官方配套，MuJoCo 动力学准，Isaac 能并行；都通过 ros2_control 插件 + 话题桥接入。
- **部署**：Docker 打包、最小镜像、多阶段构建、compose 编排、开机自启。
- **现场网络**：大消息 + Wi-Fi = 分片丢失卡 30 秒；Best effort、调内核参数、压缩图像。
- **安全**：SROS2 认证加密访问控制，联网产品要开；急停不靠 ROS。